# Try a molecule against the historical record

Nothing to install — this runs in the browser.

**Run the setup cell first.** After that every cell below is self-contained:
run them in any order, as many times as you like. The only one you need is the
second — edit the SMILES in it and run it.

### What it is actually comparing against

A molecule cannot be judged in the abstract — *"is this good?"* has no referent.
What this answers is *"is this a good optimization of **that** starting compound?"*,
and that question the published literature does settle.

The reference is **measured experimental activity** from ChEMBL: 32 real optimization
campaigns from published papers, each a starting compound plus the improved analogs
medicinal chemists actually reached, with their measured potencies.

Everything the tool reports is either a ChEMBL measurement or an RDKit substructure
match. **No model gives an opinion anywhere** — no ADMET predictor, no docking, no
LLM judge. When the published record cannot settle a question, the tool says so
instead of guessing.

## Setup — run once (takes about a minute)

In [ ]:
!pip install -q rdkit
!pip install -q git+https://github.com/SinanGokmen/leadopt-bench.git

import pandas as pd
from IPython.display import HTML, display
from rdkit import Chem
from rdkit.Chem import Draw
from leadopt_bench import metrics as M, report as R

SERIES = {s["series_id"]: s for s in M.available_series()}
print(f"ready — {len(SERIES)} series available\n")
pd.set_option("display.max_colwidth", 64)
display(pd.DataFrame(SERIES.values())[
    ["series_id", "target", "endpoint", "lead_pchembl", "n_targets", "max_gain"]])

## Test a molecule — edit and run this cell

Pick any `series_id` from the table above. The starting compound is printed for you,
so you can copy it and change something.

`MY_CHANGE` and `MY_REASONING` are optional. If you fill in the reasoning, the tool
also checks whether the structures you mention are actually present in the molecule.

In [ ]:
# ============================== EDIT THIS ==============================
SERIES_ID    = "CHEMBL6106211_IC50"
MY_SMILES    = "CC(C)(C)c1cc(NC(=O)Nc2ccc(Oc3ccnc(-c4cnn(CCCCCCC(=O)NO)c4)c3)cc2)no1"
MY_CHANGE    = "shorten the acyl chain"
MY_REASONING = "the hydroxamic acid chelates zinc; the chain should match the channel"
# =======================================================================

s = SERIES[SERIES_ID]
print(f"target:            {s['target']}")
print(f"starting compound: {s['lead_smiles']}")
print(f"its measured p{s['endpoint']}:  {s['lead_pchembl']}")
print(f"chemists reached {s['n_targets']} better compounds, best gain "
      f"{s['max_gain']} log units\n")

score = M.evaluate_smiles(MY_SMILES, SERIES_ID,
                          modification=MY_CHANGE, rationale=MY_REASONING)
display(HTML(R.verdict_html(score, s)))

mols = [Chem.MolFromSmiles(s["lead_smiles"])]
legends = [f"starting compound  (p={s['lead_pchembl']})"]
mine = Chem.MolFromSmiles(MY_SMILES)
if mine is not None:
    mols.append(mine)
    legends.append(f"yours  ({score.category})")
display(Draw.MolsToGridImage(mols, legends=legends, molsPerRow=2,
                             subImgSize=(430, 300)))

## Every verdict the tool can give

The colour reflects **how much the published record settles** — not whether a
molecule is good.

In [ ]:
display(HTML(R.legend_html()))

### The three checks behind the verdict

| Check | How it is decided | Why it is not an opinion |
|---|---|---|
| Is it a valid molecule? | RDKit parsing and sanitization, single fragment, mass range | a parser either accepts the structure or it does not |
| Was the scaffold kept? | Morgan fingerprint Tanimoto against the starting compound | the threshold is **calibrated from the series itself**: the least similar compound real chemists made from that lead |
| Is the reasoning consistent? | a 40-entry SMARTS catalogue of rings and groups; every structure the text names is looked for in the lead **or** the proposal | substructure matching is a fact about the molecule |

The third check is the mechanizable part of what a domain expert does when they
glance at a suggestion and immediately see it is about the wrong compound.

One number needs care: **change-type accuracy**. A model that only ever adds
fluorine, chlorine or a methyl scores **0.61** on it, because halogen scanning
genuinely does raise potency. So it is only readable against that null floor, which
is why the floor is printed next to the number.

## Optional: what an LLM proposed on these same series

Gemma 3 12B was asked the same question on all 32 series — 160 suggestions.
The table below is the queue that reaches a human: valid molecules that kept the
scaffold and whose reasoning is at least about the right compound, but which the
chemists never made. Only a domain expert can settle these.

In [ ]:
import importlib.resources as res

sug = pd.read_csv(res.files("leadopt_bench")
                  / "data/reports/suggestions_final_gemma3_12b.csv")

print("all 160 suggestions, by verdict:")
print(sug["category"].value_counts().to_string(), "\n")

queue = sug[(sug["category"] == "novel_plausible")
            & (~sug["describes_other_molecule"])]
print(f"reaching a human: {len(queue)} of {len(sug)} "
      f"({len(sug) - len(queue)} filtered automatically)")
queue[["target", "modification", "rationale", "smiles"]].head(10)

### What the filters threw out, and why

In [ ]:
# self-contained: reloads the table so this cell runs on its own
import importlib.resources as res
import pandas as pd

sug = pd.read_csv(res.files("leadopt_bench")
                  / "data/reports/suggestions_final_gemma3_12b.csv")

print("--- discarded: not a valid molecule ---")
print(sug[sug["category"] == "invalid"][["modification", "invalid_reason"]]
      .head(4).to_string(index=False))

n_same = (sug["category"] == "unchanged").sum()
print(f"\n--- discarded: {n_same} suggestions described a change and made none ---")

print("\n--- discarded: reasoning about structures that are not there ---")
ung = sug[sug["describes_other_molecule"] == True]
print(ung["ungrounded_claims"].value_counts().head(6).to_string())
for _, r in ung.head(2).iterrows():
    print(f"\n  claimed:  {r['modification']}")
    print(f"  not in the molecule:  {r['ungrounded_claims']}")